# 13 — Anchura de las citas: ¿la regla de citación favorece a una condición?

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`09-jss-regla-de-citacion.ipynb`](09-jss-regla-de-citacion.ipynb) ·
[ADR 0009](../docs/decisions/0009-adr-regla-de-citacion-correcta.md) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) · issue #97

---

> **Estado: análisis descriptivo sobre datos de desarrollo, como insumo para la decisión de #97.**
> Se usan las 21 preguntas de calibración de ES2015 y los registros de dos corridas de humo ya
> versionadas: C1 (`humo-c1-es2015`) y C2 con la corrección del adaptador
> (`humo-c2-es2015-conceptos`). ES2015 es una **serie de desarrollo** (ADR 0005): **ninguna cifra
> de este notebook entra en una tabla de resultados**. Con 11 a 13 respuestas por condición, el
> análisis indica una **dirección**, no es una prueba estadística. No se corrió ningún modelo
> ni ningún experimento, y **no se modifica** la regla de citación, sus cifras ni el ADR 0009:
> este notebook no decide, solo aporta evidencia.

# 1. La pregunta

El ADR 0009 declaró un riesgo: la regla de citación compara actos de diálogo, y las condiciones
citan unidades de anchura muy distinta (C1 cita fragmentos de la reunión; C2 cita conceptos y
decisiones de la ontología de OpenKOS). La regla vigente considera **correcta** una citación si
**al menos una** de sus citas comparte un acto de diálogo con la evidencia de referencia. Una
citación más ancha cubre más actos y tiene, por azar, más oportunidades de tocar la referencia.
Si una condición cita sistemáticamente más ancho, la regla podría favorecerla sin que cite mejor.

Las dos cifras que la corrida ya guarda no muestran la anchura:

- `citation_correct` solo dice si hubo al menos un acto compartido;
- `citation_precision` cuenta **citas** (piezas de la citación), no actos: una cita de 140 actos
  pesa lo mismo que una de 3.

Este notebook agrega tres medidas **solo de análisis**, definidas en
`afg.evaluation.citation_width`, todas sobre los actos de diálogo identificados por
(reunión, acto):

| Medida | Definición |
|---|---|
| **Anchura** | número de actos de diálogo distintos que contiene la citación (total por respuesta y por unidad citada) |
| **Precisión por acto** | actos citados que están en la referencia ÷ actos citados |
| **Cobertura por acto** | actos de referencia que la citación contiene ÷ actos de referencia |

Una respuesta contestada sin ninguna cita tiene precisión y cobertura **0,0**, en el mismo
espíritu del ADR 0009 (una respuesta sin fuente es una falla y no debe desaparecer del promedio).
Las medidas aplican donde aplica la regla: no a abstenciones, ni a preguntas E4, ni a preguntas
sin evidencia de referencia.

In [1]:
from dataclasses import asdict

import pandas as pd

from afg.evaluation.citation_width import (
    low_precision_hits,
    summarize_widths,
    trace_unit_widths,
    width_rows,
)
from afg.evaluation.judge_calibration_worksheet import WORKSHEET_PATH
from afg.experiments.questions import load_experiment_questions
from afg.experiments.runner import RECORDS_FILE, load_records
from afg.shared.paths import EXPERIMENT_B_DIR

pd.set_option("display.width", 170)

RUNS = {
    "C1": EXPERIMENT_B_DIR / "humo-c1-es2015",
    "C2": EXPERIMENT_B_DIR / "humo-c2-es2015-conceptos",
}
for needed in (WORKSHEET_PATH, *(run / RECORDS_FILE for run in RUNS.values())):
    if not needed.exists():
        raise SystemExit(
            f"No se encontro {needed}.\n"
            "Se generan con `uv run afg experiment b --help` (corridas de humo, requieren Ollama "
            "y, para C2, OpenKOS) y con la hoja de calibracion de ES2015."
        )

QUESTIONS = {q.id: q for q in load_experiment_questions(WORKSHEET_PATH)}
RECORDS = {c: load_records(run / RECORDS_FILE, latest_only=True) for c, run in RUNS.items()}
ROWS = {c: width_rows(records, QUESTIONS) for c, records in RECORDS.items()}
for c in RUNS:
    print(
        f"{c}: {len(RECORDS[c])} registros, {len(ROWS[c])} respuestas con evidencia de referencia"
    )

C1: 21 registros, 13 respuestas con evidencia de referencia
C2: 21 registros, 11 respuestas con evidencia de referencia


# 2. Las dos condiciones en el mismo terreno

La tabla recalcula `citation_correct` y la precisión por cita con las **funciones vigentes** para
ambas condiciones, y suma las medidas por acto. Una advertencia de lectura: los registros de C1
se escribieron con la regla original (todas las citas deben tocar la referencia, esquema v1) y
guardan `citation_correct` bajo ella (5 de 13). Aquí se recalcula con la regla vigente del ADR
0009 (al menos una cita toca), que es la que se aplica a C2; por eso C1 sube de 5 a 9.

In [2]:
STORED = {
    c: sum(
        bool(r["citation_correct"]) for r in records if r["answer"] and not r["answer"]["abstained"]
    )
    for c, records in RECORDS.items()
}
SUMMARY = {c: summarize_widths(rows) for c, rows in ROWS.items()}
pd.DataFrame(
    {
        c: {
            "respondidas (con referencia)": s.answered,
            "citación correcta, valor guardado en el registro": STORED[c],
            "citación correcta, regla vigente (ADR 0009)": s.citation_correct,
            "precisión por cita (media)": round(s.mean_unit_precision, 3),
            "precisión por acto (media)": round(s.mean_act_precision, 4),
            "cobertura por acto (media)": round(s.mean_act_recall, 3),
            "actos citados por respuesta: mín": s.cited_acts_min,
            "actos citados por respuesta: mediana": s.cited_acts_median,
            "actos citados por respuesta: máx": s.cited_acts_max,
        }
        for c, s in SUMMARY.items()
    }
)

,C1,C2
respondidas (con referencia),13.0000,11.0000
"citación correcta, valor guardado en el registro",5.0000,5.0000
"citación correcta, regla vigente (ADR 0009)",9.0000,5.0000
precisión por cita (media),0.5130,0.3030
precisión por acto (media),0.0123,0.0073
cobertura por acto (media),0.5900,0.4090
actos citados por respuesta: mín,24.0000,2.0000
actos citados por respuesta: mediana,68.0000,100.0000
actos citados por respuesta: máx,149.0000,215.0000


Las cifras guardadas coinciden con el comentario de #97 para C2 (5 correctas, precisión media
por cita 0,30). La precisión por acto es **muy baja en ambas condiciones** (del orden de un
1 %), porque la referencia es diminuta (uno a tres actos) y cualquier citación de decenas de
actos la diluye. Por eso la precisión por acto, tomada sola, no discrimina entre condiciones; lo
informativo es cuánto se cita para llegar a esos aciertos y cuánta cobertura se obtiene.

# 3. Respuesta por respuesta

Cada fila es una respuesta con evidencia de referencia. «Actos citados» y «actos de referencia»
son actos distintos; «en la referencia» cuenta los actos citados que son de referencia.

In [3]:
def table(condition):
    frame = pd.DataFrame([asdict(r) for r in ROWS[condition]])
    frame["precisión por acto"] = [r.act_precision for r in ROWS[condition]]
    frame["cobertura por acto"] = [r.act_recall for r in ROWS[condition]]
    frame["unit_widths"] = frame["unit_widths"].map(lambda w: "+".join(map(str, w)))
    return frame.rename(
        columns={
            "question": "pregunta",
            "cited_spans": "citas",
            "cited_acts": "actos citados",
            "unit_widths": "actos por cita",
            "reference_acts": "actos de referencia",
            "acts_in_reference": "en la referencia",
            "citation_correct": "citación correcta",
            "unit_precision": "precisión por cita",
        }
    ).round(3)


print("C1 (cada cita agrupa los fragmentos citados de una misma reunión)")
table("C1")

C1 (cada cita agrupa los fragmentos citados de una misma reunión)


,pregunta,citas,actos citados,actos por cita,actos de referencia,en la referencia,citación correcta,precisión por cita,precisión por acto,cobertura por acto
0,q002,3,149,48+33+68,1,1,True,0.333,0.007,1.000
1,q003,3,117,24+56+37,1,1,True,0.333,0.009,1.000
2,q004,1,24,24,1,1,True,1.000,0.042,1.000
3,q006,1,65,65,1,1,True,1.000,0.015,1.000
4,q007,2,68,25+43,1,1,True,0.500,0.015,1.000
5,q010,2,59,26+33,1,0,False,0.000,0.000,0.000
6,q011,2,52,25+27,2,0,False,0.000,0.000,0.000
7,q013,1,62,62,2,0,False,0.000,0.000,0.000
8,q017,2,67,28+39,1,0,False,0.000,0.000,0.000
9,q020,2,121,92+29,3,1,True,0.500,0.008,0.333


In [4]:
print("C2 (cada cita es un concepto o decisión, ubicado en los actos de la reunión)")
table("C2")

C2 (cada cita es un concepto o decisión, ubicado en los actos de la reunión)


,pregunta,citas,actos citados,actos por cita,actos de referencia,en la referencia,citación correcta,precisión por cita,precisión por acto,cobertura por acto
0,q003,2,98,98+4,1,0,False,0.000,0.000,0.0
1,q007,3,152,5+1+146,1,1,True,0.333,0.007,1.0
2,q008,2,100,2+98,1,1,True,0.500,0.010,1.0
3,q010,2,187,44+143,1,0,False,0.000,0.000,0.0
4,q011,1,2,2,2,0,False,0.000,0.000,0.0
5,q013,2,123,3+123,2,2,True,0.500,0.016,1.0
6,q016,2,215,69+146,1,0,False,0.000,0.000,0.0
7,q017,2,46,8+46,1,0,False,0.000,0.000,0.0
8,q023,2,76,4+76,3,3,True,1.000,0.039,1.0
9,q026,1,2,2,1,0,False,0.000,0.000,0.0


Una precisión: en el registro de respuesta (`cited_evidence`), C1 reúne en **una sola cita** todos
los fragmentos citados que son de la misma reunión, mientras que C2 entrega una cita por concepto
o decisión. Por eso «citas» y «actos por cita» no son comparables entre condiciones tal como se
guardan; la sección siguiente mide la anchura de la **unidad que el modelo realmente citó**
(un fragmento en C1, un concepto o decisión en C2), tomada de la traza de cada registro.

# 4. Anchura de la unidad citada

En C1 la unidad es el fragmento de la reunión; en C2, el concepto o la decisión de OpenKOS. Las
decisiones son enunciados puntuales y se separan de los demás conceptos (descripciones de
tema, eventos, proyectos), que abarcan tramos amplios.

In [5]:
UNITS = []
for c, records in RECORDS.items():
    for record in records:
        if record["answer"] and not record["answer"]["abstained"]:
            for unit, width in trace_unit_widths(record).items():
                kind = (
                    "decisión"
                    if unit.startswith("decisions/")
                    else ("fragmento" if c == "C1" else "concepto no decisión")
                )
                UNITS.append({"condición": c, "tipo": kind, "unidad": unit, "actos": width})
UNITS = pd.DataFrame(UNITS)
(
    UNITS.groupby(["condición", "tipo"])["actos"]
    .agg(unidades="count", mínimo="min", mediana="median", máximo="max")
    .round(1)
)

unidades  mínimo  mediana  máximo
condición tipo                                                   
C1        fragmento                   35      16     27.0      47
C2        concepto no decisión        11      44     98.0     146
          decisión                     9       1      3.0       8

In [6]:
BINS = [0, 10, 20, 30, 50, 100, 1000]
LABELS = ["1-10", "11-20", "21-30", "31-50", "51-100", "más de 100"]
UNITS["tramo de anchura (actos)"] = pd.cut(UNITS["actos"], BINS, labels=LABELS)
pd.crosstab(UNITS["tramo de anchura (actos)"], [UNITS["condición"], UNITS["tipo"]])

condición                       C1                   C2         
tipo                     fragmento concepto no decisión decisión
tramo de anchura (actos)                                        
1-10                             0                    0        9
11-20                            3                    0        0
21-30                           17                    0        0
31-50                           15                    2        0
51-100                           0                    4        0
más de 100                       0                    5        0

Los fragmentos de C1 son homogéneos (de 16 a 47 actos, mediana 27). C2 es bimodal: sus decisiones
son estrechas (de 1 a 8 actos) y sus demás conceptos son anchos (de 44 a 146 actos), con los más
anchos por encima de cualquier fragmento de C1.

# 5. Anchura de la citación completa y su relación con el acierto

Lo que la regla evalúa es la citación **completa** de la respuesta: todos los actos que la
suman. La tabla separa las respuestas con citación correcta de las demás.

In [7]:
FULL = pd.DataFrame(
    [
        {
            "condición": c,
            "citación correcta": r.citation_correct,
            "actos citados": r.cited_acts,
            "precisión por acto": r.act_precision,
            "cobertura por acto": r.act_recall,
        }
        for c, rows in ROWS.items()
        for r in rows
    ]
)
(
    FULL.groupby(["condición", "citación correcta"])
    .agg(
        respuestas=("actos citados", "count"),
        mediana_actos=("actos citados", "median"),
        mín_actos=("actos citados", "min"),
        máx_actos=("actos citados", "max"),
        precisión_acto=("precisión por acto", "mean"),
        cobertura_acto=("cobertura por acto", "mean"),
    )
    .round(3)
)

respuestas  mediana_actos  mín_actos  máx_actos  precisión_acto  cobertura_acto
condición citación correcta                                                                                 
C1        False                       4           60.5         52         67           0.000           0.000
          True                        9           85.0         24        149           0.018           0.852
C2        False                       6           72.0          2        215           0.000           0.000
          True                        5          123.0         76        152           0.016           0.900

# 6. Vista de control por anchura

Pregunta: ¿cuántas citaciones «correctas» lo son con una precisión por acto casi nula? Se
muestra la distribución completa de la precisión por acto de las citaciones correctas y, a modo
de referencia, el conteo bajo tres umbrales (5 %, 10 % y 20 %). No hay un umbral natural, por lo
que se muestran varios en lugar de elegir uno.

In [8]:
print("Precisión por acto de las citaciones correctas (ordenadas):")
for c, rows in ROWS.items():
    values = sorted(round(r.act_precision, 3) for r in rows if r.citation_correct)
    print(f"  {c}: {values}")
print()
pd.DataFrame(
    {
        f"precisión por acto < {t:.0%}": {
            c: f"{low_precision_hits(rows, threshold=t)} de {SUMMARY[c].citation_correct}"
            for c, rows in ROWS.items()
        }
        for t in (0.05, 0.10, 0.20)
    }
)

Precisión por acto de las citaciones correctas (ordenadas):
  C1: [0.007, 0.008, 0.009, 0.014, 0.015, 0.015, 0.024, 0.027, 0.042]
  C2: [0.007, 0.008, 0.01, 0.016, 0.039]



,precisión por acto < 5%,precisión por acto < 10%,precisión por acto < 20%
C1,9 de 9,9 de 9,9 de 9
C2,5 de 5,5 de 5,5 de 5


Con una referencia de uno a tres actos y citaciones de decenas a cientos de actos, la
precisión por acto de **toda** citación correcta es inferior al 5 % en ambas condiciones, así
que esta vista no separa a las condiciones: el umbral no distingue. La señal está en la
anchura y la cobertura (secciones 4 y 5), no en la precisión por acto.

# 7. Lectura

Lectura descriptiva, sin decidir, con las salvedades de la sección 8.

- **Citación por respuesta.** Ambas condiciones citan en promedio decenas de actos por
  respuesta; la mediana de C2 (100) es mayor que la de C1 (68) y su máximo también (215 frente a
  149). Con tan pocas respuestas esa diferencia no es concluyente.
- **Dónde está la anchura de C2.** Se concentra en los conceptos que no son decisiones (44 a
  146 actos). Sus decisiones son más estrechas que cualquier fragmento de C1. Ninguna
  condición cita de forma puntual en la mayor parte de sus respuestas.
- **Aciertos y anchura.** En ambas condiciones las citaciones correctas son más anchas que
  las incorrectas: C1, mediana de 85 actos frente a 60,5; C2, 123 frente a 72 (sección 5). Es
  la señal que cabría esperar si la anchura ayudara a tocar una referencia de uno a tres actos,
  y es más marcada en C2: sus cuatro aciertos con conceptos amplios (q007, q008, q013, q029) se
  dan con un concepto de 98 a 146 actos, y sus dos citaciones más estrechas (q011 y q026, de
  2 actos cada una) fallan. En C1 no hay citaciones estrechas con qué contrastar (todas tienen
  al menos 24 actos). Este notebook no determina qué unidad de cada respuesta aportó el acierto.
- **Cobertura.** La cobertura media de C1 (0,59) es mayor que la de C2 (0,41), aunque C2 cita
  más actos en total. Cuando la citación acierta, ambas contienen casi toda la referencia
  (0,85 en C1 y 0,90 en C2).
- **Contra la lectura de que C2 se beneficia.** C2 cita más ancho que C1 y aun así acierta
  menos (5 de 11 frente a 9 de 13 con la regla vigente). La anchura no alcanza para explicar
  las diferencias de acierto entre condiciones en esta muestra.

Como insumo para #97: la anchura es un posible factor de confusión de `citation_correct` para
**ambas** condiciones, y se asocia al acierto sobre todo en C2; esta muestra no permite afirmar
que la regla favorezca a C2 sobre C1. Una medida que la neutralice (por ejemplo, exigir acierto en una unidad citada
estrecha, o ponderar por anchura) es una decisión de diseño que corresponde a #97.

# 8. Qué no permite concluir este análisis

- Son 13 respuestas de C1 y 11 de C2 en una sola serie de desarrollo, con un único modelo
  generador y una sola corrida por condición. Las cifras describen estas corridas; no estiman
  nada ni admiten una prueba estadística.
- Las preguntas contestadas por una y otra condición no son las mismas (la abstención difiere),
  por lo que las medias no comparan lo mismo pregunta a pregunta.
- La evidencia de referencia es escasa (uno a tres actos, de un solo anotador) y fija el techo de
  la precisión por acto. Con la anotación adjudicada de las series de evaluación podría cambiar.
- La anchura de C2 depende de cómo el adaptador ubica cada concepto en la transcripción
  (T10 de #95). Las citas a `sources/<reunión>`, que contienen la transcripción completa, quedan
  fuera de `cited_evidence`; este análisis no las evalúa.
- Las medidas por acto no sustituyen a `citation_correct` ni a la precisión por cita; son
  lecturas complementarias. Ninguna de estas cifras es un resultado de la tesis (ADR 0005).